## Generating vector embeddings for MD files with milvus

### Hybrid model

In [ ]:
from pymilvus import MilvusClient

embedded_milvus = True

if embedded_milvus:
    client = MilvusClient("milvus_db.db")
else:
    client = MilvusClient(host="localhost", port=1953, timeout=10)

In [2]:
# delete schema
if client.has_collection(collection_name="hybrid_collection"):
    client.drop_collection(collection_name="hybrid_collection")

In [ ]:
from pymilvus import MilvusClient, DataType

# Define schema
schema = MilvusClient.create_schema()

# Add fields to schema
schema.add_field(field_name="id",
                 datatype=DataType.INT64,
                 is_primary=True)
if embedded_milvus:
    schema.add_field(field_name="text",
                     datatype=DataType.VARCHAR)
schema.add_field(field_name="dense_vector",
                 datatype=DataType.FLOAT_VECTOR,
                 dim=768)
schema.add_field(field_name="sparse_vector",
                 datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="availability_categories",
                 datatype=DataType.ARRAY,
                 element_type=DataType.VARCHAR,
                 max_length=5,
                 max_capacity=5)

# Prepare index parameters
index_params = client.prepare_index_params()
index_params.add_index(field_name="dense_vector",
                       index_type="FLAT",
                       metric_type="COSINE",
                       index_name="dense_index")
index_params.add_index(field_name="sparse_vector",
                       index_type="SPARSE_INVERTED_INDEX",
                       metric_type="IP",
                       index_name="sparse_index")

if client.has_collection(collection_name="hybrid_collection"):
    client.drop_collection(collection_name="hybrid_collection")
client.create_collection(
    collection_name="hybrid_collection",
    schema=schema,
    index_params=index_params)


In [4]:
# Chekk if the collection has correct indexes
res = client.list_indexes(
    collection_name="hybrid_collection"
)

print(res)

['dense_index', 'sparse_index']


In [5]:
# Wczytranie kategorii
import pandas as pd
df_cat = pd.read_csv("Rejestr_Produktow_Leczniczych_z_kategoriami.csv", sep=";", encoding="utf-8")
df_cat.head(50)
df_cat["Kategorie dostępności"].unique()


array(['Rpz', 'Rp', 'OTC', 'OTC, Rp', 'Lz', 'Lz, Rp', 'Rp, Rpw', 'Rpw',
       'Lz, OTC', 'Rp, Rpz', nan], dtype=object)

### With chunking

In [5]:
import os
import re
import glob

def extract_categories_and_chunks(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Extract ID from the filename
    id_match = re.search(r'char_(\d+)', file_path)
    if not id_match:
        raise ValueError(f"ID not found in file name: {file_path}")

    id_value = id_match.group(1)

    # Extract categories to list of strings from text separated by commas
    raw_categories = df_cat.loc[df_cat["Identyfikator Produktu Leczniczego"] == id_value]["Kategorie dostępności"]
    if raw_categories.empty:
        categories = []
    else:
        raw_categories = raw_categories.values[0]
        if isinstance(raw_categories, str):
            categories = [cat.strip() for cat in raw_categories.split(",")]
        else:
            categories = []

    # This regex pattern captures the product name and any additional information
    # meta_match = re.search(r"## 1\. NAZWA PRODUKTU LECZNICZEGO\s+(.*?)\n", content, re.IGNORECASE | re.DOTALL)
    # metadata = meta_match.group(1).strip() if meta_match else "Unknown"

    # Chunk content
    content = content[1:].strip()
    chunks = [chunk.strip() for chunk in content.split("\n\n") if chunk.strip()]

    return categories, chunks, id_value

# Process all Markdown files
markdown_folder = "charakterystyki_md"
file_paths = glob.glob(os.path.join(markdown_folder, "*.md"))[:7]

all_texts = []
all_categories = []
all_ids = []

for file_path in file_paths:
    categories, chunks, id_value = extract_categories_and_chunks(file_path)
    all_texts.extend(chunks)
    all_categories.extend(categories)
    all_ids.extend(id_value)
    # all_metadata.extend([metadata] * len(chunks))

### Wtihout chunking

In [10]:
import os
import re
import glob

def extract_categories_and_text(file_path, df_cat):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Extract ID from the filename
    id_match = re.search(r'char_(\d+)', file_path)
    if not id_match:
        raise ValueError(f"ID not found in file name: {file_path}")

    id_value = id_match.group(1)

    try:
        id_value = int(id_value)
    except ValueError:
        raise ValueError(f"ID is not an integer: {id_value}")

    # Extract categories to list of strings from text separated by commas
    raw_categories = df_cat.loc[df_cat["Identyfikator Produktu Leczniczego"] == id_value]["Kategorie dostępności"]
    if raw_categories.empty:
        categories = []
    else:
        raw_categories = raw_categories.values[0]
        if isinstance(raw_categories, str):
            categories = [cat.strip() for cat in raw_categories.split(",")]
        else:
            categories = []

    return categories, content, id_value

markdown_folder = "charakterystyki_md"
file_paths = glob.glob(os.path.join(markdown_folder, "*.md"))[:20]

all_ids = []
all_texts = []
all_categories = []

for file_path in file_paths:
    categories, text, id_value = extract_categories_and_text(file_path, df_cat)
    all_texts.append(text)
    all_categories.append(categories)
    all_ids.append(id_value)


In [11]:
all_ids

[100132508,
 100280077,
 100021312,
 100330846,
 100366347,
 100111009,
 100425110,
 100063871,
 100436504,
 100401820,
 100470793,
 100412700,
 100010604,
 100146670,
 100480809,
 100401718,
 100396696,
 100211811,
 100427646,
 100438087]

In [12]:
from pymilvus.model.hybrid import MGTEEmbeddingFunction

# m2v_ef = model.dense.Model2VecEmbeddingFunction(
#     model_source='minishlab/potion-base-8M', #
# )
mgte_ef = MGTEEmbeddingFunction(
    model_name="Alibaba-NLP/gte-multilingual-base",
)

vectors = mgte_ef.encode_documents(all_texts)

In [15]:
mgte_ef.dim # (768, 25002)


{'dense': 768, 'sparse': 250002}

In [ ]:
####################################################


In [ ]:
# Prepare data for insertion
data = []

for idx, text in enumerate(all_texts):
    # Extract indices and values from the sparse vector (assumes sparse vectors are in COO format)
    indices = vectors['sparse'][idx].col.tolist()  # Access the column indices (non-zero positions)
    values = vectors['sparse'][idx].data.tolist()  # Access the non-zero values

    # Build the sparse vector entry
    sparse_vector_data = [(indices[i], values[i]) for i in range(len(indices))]

    # Prepare the dictionary for each entry
    if embedded_milvus
        entry = {
            "id": all_ids[idx],
            "text": text,
            "dense_vector": vectors['dense'][idx].tolist(),  # Ensure this is a list of floats
            "sparse_vector": sparse_vector_data,
            "availability_categories": all_categories[idx]
        }
    else:
        entry = {
            "id": all_ids[idx],
            "dense_vector": vectors['dense'][idx].tolist(),  # Ensure this is a list of floats
            "sparse_vector": sparse_vector_data,
            "availability_categories": all_categories[idx]
        }
    # Append the entry to the data list
    data.append(entry)

#display the first entry sparse vector
# print("Sparse vector entry:", data[0]['sparse_vector'])
# print(data[0])
#
# Now insert the data into Milvus
client.insert(
    collection_name="hybrid_collection",
    data=data  # Directly pass the list of dictionaries
)

print("✅ Data inserted successfully into Milvus.")

✅ Data inserted successfully into Milvus.


In [17]:
print(data[0]['sparse_vector'])

[(4, 0.2557430863380432), (5, 0.35397180914878845), (6, 0.3994766175746918), (8, 0.19807429611682892), (9, 0.5102496147155762), (11, 0.23679596185684204), (12, 0.143375426530838), (13, 0.299111008644104), (14, 0.18105585873126984), (15, 0.5861276984214783), (16, 0.19620376825332642), (17, 0.11248840391635895), (19, 0.39003315567970276), (20, 0.24460428953170776), (21, 0.24284076690673828), (23, 0.13646745681762695), (28, 0.6539291143417358), (34, 0.43355077505111694), (47, 0.18245041370391846), (48, 0.5833166241645813), (55, 0.3291792869567871), (62, 0.010108530521392822), (64, 0.5945138335227966), (66, 0.680282473564148), (68, 0.15312308073043823), (73, 0.12787139415740967), (75, 0.22802066802978516), (76, 0.12569066882133484), (78, 0.2754048705101013), (80, 0.18268468976020813), (82, 0.927442193031311), (93, 0.10881008207798004), (95, 0.9688855409622192), (97, 0.3156837224960327), (107, 0.13708767294883728), (108, 0.9011301398277283), (133, 0.3007059097290039), (138, 0.40177816152572

In [18]:
# query the first entry in the collection
results = client.search(
    collection_name="hybrid_collection",
    data=[data[0]['dense_vector']],  # Query with the dense vector of the first entry
    anns_field="dense_vector",
    params={"metric_type": "IP"},  # Use Inner Product (IP) for similarity
    limit=1,  # Retrieve the top 1 closest result
    output_fields=["id", "dense_vector"],  # specifies fields to be returned
)
print(results)

data: [[{'id': 100132508, 'distance': 1.0, 'entity': {'dense_vector': [-0.030030813068151474, 0.097610242664814, 0.0008154097013175488, 0.08880365639925003, 0.03834274038672447, -0.05521366000175476, -0.01719474047422409, -0.010473302565515041, 0.03186734393239021, -0.13739976286888123, 0.012652882374823093, 0.05714865401387215, 0.004346460569649935, 0.06009656563401222, -0.07834780961275101, 0.09596975892782211, 0.07476502656936646, 0.048985596746206284, 0.06163923814892769, 0.03121034801006317, 0.11028879135847092, 0.03515045344829559, 0.038658980280160904, 0.052822958678007126, 0.07547590136528015, 0.05315957963466644, 0.045094139873981476, -0.0347389280796051, 0.03423362970352173, 0.04914109781384468, -0.024012604728341103, -0.0751805230975151, 0.014255261979997158, 0.026009751483798027, 0.12070566415786743, 0.010216369293630123, 0.05368015170097351, -0.026697102934122086, -0.07511251419782639, -0.037315357476472855, -0.0013966752449050546, -0.0036503197625279427, 0.026228645816445

In [19]:
from pymilvus import AnnSearchRequest
from pymilvus import RRFRanker

def query_hybrid_search(query: str):
    query_embeddings = mgte_ef([query])

    sparse_req = AnnSearchRequest(
        query_embeddings["sparse"], "sparse_vector", param={"metric_type": "IP"}, limit=10
    )
    dense_req = AnnSearchRequest(
        query_embeddings["dense"], "dense_vector", {"metric_type": "COSINE"}, limit=10
    )

    result = client.hybrid_search(
        collection_name="hybrid_collection", reqs=[sparse_req, dense_req], ranker=RRFRanker(), limit=2, output_fields=["id"]
    )

    return result


In [20]:
res = client.list_indexes(
    collection_name="hybrid_collection"
)

print(res)

['dense_index', 'sparse_index']


In [21]:
res = client.describe_index(
    collection_name="hybrid_collection",
    index_name="sparse_index"
)

print(res)

{'metric_type': 'IP', 'index_type': 'SPARSE_INVERTED_INDEX', 'field_name': 'sparse_vector', 'index_name': 'sparse_index', 'total_rows': 0, 'indexed_rows': 0, 'pending_index_rows': 0, 'state': 'Finished'}


In [26]:
query_hybrid_search("""biały proszek do wstrzykiwań wykorzystywany z zakresu onkologii i hematologii""")


data: [[{'id': 100132508, 'distance': 0.03109932318329811, 'entity': {'id': 100132508}}, {'id': 100480809, 'distance': 0.03077651560306549, 'entity': {'id': 100480809}}]]

In [27]:
def extract_ids(data):
    """
    Extracts 'id' values from a nested list of result dictionaries.

    Args:
        data (list): A nested list in the form [[{...}, {...}, ...]].

    Returns:
        list: A flat list of IDs.
    """
    return [item['id'] for sublist in data for item in sublist]


In [28]:
res = query_hybrid_search("""biały proszek do wstrzykiwań wykorzystywany z zakresu onkologii i hematologii""")
extract_ids(res)

[100132508, 100480809]

In [23]:
def query_dense_search(query: str):
    query_embeddings = mgte_ef([query])
    dense_vec = query_embeddings['dense'][0].tolist()

    search_params = {
        "collection_name": "hybrid_collection",
        "data": [dense_vec],
        "anns_field": "dense_vector",
        "params":{},
        "metric_type": "COSINE",
        "limit": 2,
        "output_fields": ["id"],
    }
    res_dense = client.search(**search_params)[0]

    return res_dense

In [21]:
for field in schema.fields:
    print(f"Field name: {field.name}, type: {field.dtype}, params: {field.params}")

# Print out each field and its properties
for field in schema.fields:
    if field.name == "dense_vector":
        print(f"Dense vector dimension: {field.params['dim']}")

Field name: id, type: 5, params: {}
Field name: text, type: 21, params: {'max_length': 65535}
Field name: dense_vector, type: 101, params: {'dim': 768}
Field name: sparse_vector, type: 104, params: {'dim': 250002}
Field name: availability_categories, type: 22, params: {'max_length': 5}
Dense vector dimension: 768


In [25]:
query_dense_search("""biały proszek do wstrzykiwań wykorzystywany z zakresu onkologii i hematologii""")

[{'id': 100132508, 'distance': 0.6110409498214722, 'entity': {'id': 100132508}}, {'id': 100146670, 'distance': 0.5875734686851501, 'entity': {'id': 100146670}}]

# ---------------------------------------------------------